In [1]:
%pip install -q dotenv llama_stack_client==0.4.2

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import requests
from io import BytesIO
from dotenv import load_dotenv

from llama_stack_client import LlamaStackClient

In [3]:
load_dotenv()
base_url = os.getenv("REMOTE_BASE_URL", "http://localhost:8321")

client = LlamaStackClient(base_url=base_url)

models = client.models.list()
[m for m in models if m.custom_metadata.get("model_type") == "embedding"]

INFO:httpx:HTTP Request: GET http://llamastack-distribution-service:8321/v1/models "HTTP/1.1 200 OK"


[Model(id='sentence-transformers/ibm-granite/granite-embedding-125m-english', created=1772758271, owned_by='llama_stack', custom_metadata={'model_type': 'embedding', 'provider_id': 'sentence-transformers', 'provider_resource_id': 'ibm-granite/granite-embedding-125m-english', 'embedding_dimension': 768}, object='model'),
 Model(id='sentence-transformers/nomic-ai/nomic-embed-text-v1.5', created=1772758271, owned_by='llama_stack', custom_metadata={'model_type': 'embedding', 'provider_id': 'sentence-transformers', 'provider_resource_id': 'nomic-ai/nomic-embed-text-v1.5', 'embedding_dimension': 768}, object='model')]

In [4]:
embedding_model = os.getenv("VDB_EMBEDDING", "sentence-transformers/ibm-granite/granite-embedding-125m-english")
embedding_dimension = int(os.getenv("VDB_EMBEDDING_DIMENSION", 768))

vs = client.vector_stores.create(
    name="hr-benefits-hybrid",
    extra_body={
        "embedding_model": embedding_model,
        "embedding_dimension": embedding_dimension,
        "search_mode": "hybrid",
        "bm25_weight": 0.5,
        "semantic_weight": 0.5,
    }
)

INFO:httpx:HTTP Request: POST http://llamastack-distribution-service:8321/v1/vector_stores "HTTP/1.1 200 OK"


In [5]:
url = "https://raw.githubusercontent.com/burrsutter/fantaco-redhat-one-2026/refs/heads/main/rag-llama-stack/source_docs/FantaCoFabulousHRBenefits_clean.txt"

response = requests.get(url, timeout=30)
text_content = response.text

In [6]:
text_buffer = BytesIO(text_content.encode('utf-8'))
text_buffer.name = "hr-benefits-clean.txt"

uploaded_file = client.files.create(
    file=text_buffer,
    purpose="assistants"
)

INFO:httpx:HTTP Request: POST http://llamastack-distribution-service:8321/v1/files "HTTP/1.1 200 OK"


In [7]:
client.vector_stores.files.create(
    vector_store_id=vs.id,
    file_id=uploaded_file.id,
    chunking_strategy={
        "type": "static",
        "static": {
            "max_chunk_size_tokens": 100,
            "chunk_overlap_tokens": 10
        }
    }
)

INFO:httpx:HTTP Request: POST http://llamastack-distribution-service:8321/v1/vector_stores/vs_d48c5b7b-1b95-49a4-98b0-aaa2da5cfc78/files "HTTP/1.1 200 OK"


VectorStoreFile(id='file-fcf3acbcb1234aa0a1b36a87d3d86338', chunking_strategy=ChunkingStrategyVectorStoreChunkingStrategyStatic(static=ChunkingStrategyVectorStoreChunkingStrategyStaticStatic(chunk_overlap_tokens=10, max_chunk_size_tokens=100), type='static'), created_at=1772758288, status='completed', vector_store_id='vs_d48c5b7b-1b95-49a4-98b0-aaa2da5cfc78', attributes={}, last_error=None, object='vector_store.file', usage_bytes=0)

In [8]:
MODEL = "vllm/qwen3-8b"
INSTRUCTIONS = "You MUST use the knowledge_search tool to answer ALL questions by searching the provided documents."
VECTOR_STORE_ID = vs.id

TOOLS = [
    {
        "type": "function",
        "name": "knowledge_search",
        "description": "Search the HR benefits knowledge base for relevant information",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "The search query to find relevant documents"}
            },
            "required": ["query"],
        },
    }
]

In [9]:
def execute_rag_tool_call(response):
    """Execute knowledge_search function calls via vector_stores.search."""
    function_calls = []
    for item in response.output:
        if getattr(item, "type", None) == "function_call":
            function_calls.append(item)

    if not function_calls:
        return None

    import json
    tool_inputs = []
    for fc in function_calls:
        args = json.loads(fc.arguments) if isinstance(fc.arguments, str) else fc.arguments
        query = args.get("query", "")
        print(f"\n🔍 Searching: {query}")

        search_response = client.vector_stores.search(
            vector_store_id=VECTOR_STORE_ID,
            query=query,
            max_num_results=5,
        )
        results_text = "\n".join(
            f"[{i+1}] {r.content[0].text}" for i, r in enumerate(search_response.data) if r.content
        )
        print(f"📋 Found {len(search_response.data)} results")

        tool_inputs.append({
            "type": "function_call_output",
            "call_id": fc.call_id,
            "output": results_text if results_text else "No results found.",
        })

    return client.responses.create(
        model=MODEL,
        input=tool_inputs,
        instructions=INSTRUCTIONS,
        tools=TOOLS,
        stream=True,
        previous_response_id=response.id,
    )


def stream_rag(stream):
    """Stream responses with RAG tool call handling."""
    final_response = None

    for event in stream:
        event_type = getattr(event, "type", None)

        if event_type == "response.output_text.delta":
            print(event.delta, end="", flush=True)
        elif event_type == "response.refusal.delta":
            print(event.delta, end="", flush=True)
        elif event_type == "response.completed":
            final_response = event.response

    print()

    if final_response:
        next_stream = execute_rag_tool_call(final_response)
        if next_stream:
            stream_rag(next_stream)


query = "What do I receive when I retire?"

stream = client.responses.create(
    model=MODEL,
    input=query,
    instructions=INSTRUCTIONS,
    tools=TOOLS,
    stream=True,
)

stream_rag(stream)

INFO:httpx:HTTP Request: POST http://llamastack-distribution-service:8321/v1/responses "HTTP/1.1 200 OK"


<think>
Okay, the user is asking, "What do I receive when I retire?" I need to figure out what information to provide. Since they're asking about retirement benefits, I should use the knowledge_search tool to look up relevant documents.

First, I'll check the available functions. The knowledge_search function is designed to search the HR benefits knowledge base. The parameter required is a query string. The user's query is about retirement benefits, so the query should be something like "retirement benefits" or "what do I receive when I retire."

I need to make sure the search term is specific enough to find the right documents. Maybe using exact phrases from the user's question would yield better results. Let me structure the function call with the query parameter set to "What do I receive when I retire?" to match their exact wording. That way, the search can find documents that directly address their question. 

I'll format the tool call as specified, using JSON inside the tool_call 

INFO:httpx:HTTP Request: POST http://llamastack-distribution-service:8321/v1/vector_stores/vs_d48c5b7b-1b95-49a4-98b0-aaa2da5cfc78/search "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://llamastack-distribution-service:8321/v1/responses "HTTP/1.1 200 OK"




🔍 Searching: What do I receive when I retire?
📋 Found 5 results
<think>
Okay, the user asked, "What do I receive when I retire?" Let me check the documents provided.

Looking at the search results, they all seem to be from the same source, maybe a company's benefits handbook. The first document mentions a 401(k) that's tied to astrological alignments and positive affirmations, which sounds fictional. It also talks about a chocolate statue and a personal bard. The second document refers to the "Midas Touch & Beyond" Retirement Plan, which includes the 401(k) details. The third and fourth documents mention things like Dolphin and Squirrel phrases, alchemists, and the office griffin, which seem like quirky rules rather than retirement benefits. The fifth document praises the user's skills and mentions the benefits package being "ridiculously wonderful" but doesn't list specific items.

Putting this together, the main retirement benefits mentioned are the 401(k) with the unique condition